In [9]:
# Cell 1: environment and paths

import os
import sys
from pathlib import Path

import numpy as np
import torch
from omegaconf import OmegaConf

# Project root (adjust if you run this from a different path)
PROJECT_ROOT = Path("/root/drivestudio-coding").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
print(f"Project root: {PROJECT_ROOT}")


Using device: cuda
Project root: /root/drivestudio-coding


In [10]:
# Cell 2: config and overfit batch paths

config_path = PROJECT_ROOT / "configs/minimal_streetforward.yaml"
overfit_batch_path = PROJECT_ROOT / "data/overfit_batches/scene0_seg0_batch.pt"

print(f"Config path: {config_path}")
print(f"Overfit batch path: {overfit_batch_path}")

cfg = OmegaConf.load(str(config_path))

print("Model bbox / voxel_size / sh_degree:")
print("  bbx_min:", cfg.model.get("bbx_min"))
print("  bbx_max:", cfg.model.get("bbx_max"))
print("  voxel_size:", cfg.model.get("voxel_size"))
print("  sh_degree:", cfg.model.get("sh_degree"))


Config path: /root/drivestudio-coding/configs/minimal_streetforward.yaml
Overfit batch path: /root/drivestudio-coding/data/overfit_batches/scene0_seg0_batch.pt


Model bbox / voxel_size / sh_degree:
  bbx_min: [-40.0, -20.0, -20.0]
  bbx_max: [40.0, 4.8, 70.0]
  voxel_size: 0.2
  sh_degree: 1


In [11]:
# Cell 3: load raw overfit batch and sanity check

from tools.overfit_one_batch import load_batch

raw_batch = load_batch(str(overfit_batch_path))
print("Raw batch keys:", list(raw_batch.keys()))

pointcloud = raw_batch.get("pointcloud", None)
if pointcloud is None:
    raise ValueError("Batch has no 'pointcloud' key")

if isinstance(pointcloud, dict):
    background = pointcloud.get("background", None)
    if background is None:
        print("WARNING: pointcloud has no 'background' key")
    else:
        print("background shape:", background.shape)
        if background.size > 0:
            coords_np = background[:, :3]
            colors_np = background[:, 3:]
            print("coords min:", coords_np.min(axis=0))
            print("coords max:", coords_np.max(axis=0))
            if colors_np.size > 0:
                print("colors min:", colors_np.min(axis=0))
                print("colors max:", colors_np.max(axis=0))
else:
    print("pointcloud type:", type(pointcloud))

target_data = raw_batch.get("target", raw_batch.get("targets"))
if target_data is None:
    print("No 'target' or 'targets' found in batch")
else:
    if isinstance(target_data, dict):
        num_targets = int(target_data["image"].shape[0])
        print("num_targets (dict):", num_targets)
        print("target image shape:", target_data["image"].shape)
    else:
        print("targets list length:", len(target_data))
        if len(target_data) > 0:
            first = target_data[0]
            img = first.get("gt_image") if isinstance(first, dict) else None
            if img is not None:
                print("first target image shape:", tuple(img.shape))


Raw batch keys: ['scene_id', 'scene_folder_name', 'segment_id', 'segment_first_pose', 'segment_first_frame_idx', 'segment_first_pose_source', 'keyframe_info', 'source', 'target', 'pointcloud', 'dynamic_info', 'test']
background shape: (212417, 6)
coords min: [-19.99611    -7.1535707  -1.964318 ]
coords max: [19.999937   4.799998   6.2063003]
colors min: [0. 0. 0.]
colors max: [248. 229. 237.]
num_targets (dict): 18
target image shape: torch.Size([18, 300, 533, 3])


(viser) Connection opened (0, 1 total), 5 persistent messages

In [8]:
# Optional: web visualize raw pointcloud using PLYViewer (similar to StreetForward_Demo)

if 'pointcloud' in locals() and pointcloud is not None:
    from tools.plyviewer import PLYViewer
    import numpy as np

    demo_pointcloud = pointcloud

    viewer = PLYViewer(
        host="0.0.0.0",
        port=7007,
        point_size=0.01,
        point_shape="circle",
        auto_fallback=True,
    )

    viewer.start_viewer()

    # background cloud
    background = demo_pointcloud.get("background", np.zeros((0, 6), dtype=np.float32))
    if len(background) > 0:
        background_points = background[:, :3]
        background_colors = background[:, 3:]

        max_points_for_viz = 100000
        if len(background_points) > max_points_for_viz:
            indices = np.random.choice(len(background_points), max_points_for_viz, replace=False)
            background_points = background_points[indices]
            background_colors = background_colors[indices]

        viewer.add_point_cloud(
            points=background_points,
            colors=background_colors,
            name="/background",
            visible=True,
        )
        print(f"  ✓ added background cloud: {len(background_points):,} points")

    # dynamic objects if present
    dynamic = demo_pointcloud.get("dynamic", {}) if isinstance(demo_pointcloud, dict) else {}
    if len(dynamic) > 0:
        for intid, pts in dynamic.items():
            if len(pts) == 0:
                continue
            dynamic_points = pts[:, :3]
            dynamic_colors = pts[:, 3:]

            max_dynamic_points = 10000
            if len(dynamic_points) > max_dynamic_points:
                indices = np.random.choice(len(dynamic_points), max_dynamic_points, replace=False)
                dynamic_points = dynamic_points[indices]
                dynamic_colors = dynamic_colors[indices]

            viewer.add_point_cloud(
                points=dynamic_points,
                colors=dynamic_colors,
                name=f"/dynamic/object_{intid}",
                visible=True,
            )

    print("\n", viewer.viewer_info)
    demo_viewer = viewer
else:
    print("No 'pointcloud' variable found; run Cell 3 first.")


[21:28:40] Port 7007 is already in use. Attempting to free the port (only viewer processes)...     ]8;id=693167;file:///root/drivestudio-coding/tools/plyviewer.py\plyviewer.py]8;;\:]8;id=583444;file:///root/drivestudio-coding/tools/plyviewer.py#264\264]8;;\

           Could not check processes on port 7007: [Errno 2] No such file or directory: 'lsof'     ]8;id=199149;file:///root/drivestudio-coding/tools/plyviewer.py\plyviewer.py]8;;\:]8;id=649502;file:///root/drivestudio-coding/tools/plyviewer.py#247\247]8;;\

           Could not free port 7007 (may be used by other processes). Automatically finding an     ]8;id=683479;file:///root/drivestudio-coding/tools/plyviewer.py\plyviewer.py]8;;\:]8;id=727459;file:///root/drivestudio-coding/tools/plyviewer.py#279\279]8;;\
           available port...                                                                                       

           Using port 46751 instead. Access viewer at: http://localhost:46751                      ]8;id=421029;file:///root/drivestudio-coding/tools/plyviewer.py\plyviewer.py]8;;\:]8;id=413648;file:///root/drivestudio-coding/tools/plyviewer.py#284\284]8;;\

           Starting viewer server on 0.0.0.0:46751                                                 ]8;id=789780;file:///root/drivestudio-coding/tools/plyviewer.py\plyviewer.py]8;;\:]8;id=978067;file:///root/drivestudio-coding/tools/plyviewer.py#370\370]8;;\

╭────── viser (listening *:46751) ───────╮
│             ╷                          │
│   HTTP      │ http://localhost:46751   │
│   Websocket │ ws://localhost:46751     │
│             ╵                          │
╰────────────────────────────────────────╯

           Viewer running locally at: http://localhost:46751 (listening on 0.0.0.0)                ]8;id=472525;file:///root/drivestudio-coding/tools/plyviewer.py\plyviewer.py]8;;\:]8;id=386252;file:///root/drivestudio-coding/tools/plyviewer.py#379\379]8;;\

           Press Ctrl+C to stop the viewer                                                         ]8;id=939757;file:///root/drivestudio-coding/tools/plyviewer.py\plyviewer.py]8;;\:]8;id=845146;file:///root/drivestudio-coding/tools/plyviewer.py#380\380]8;;\


 Viewer running locally at: http://localhost:46751 (listening on 0.0.0.0)


In [ ]:
# Cell 4: convert raw batch to MinimalStreetForward format

from tools.train_minimal_streetforward import convert_batch_to_minimal_format

minimal_batch = convert_batch_to_minimal_format(raw_batch, device=device)

print("Minimal batch keys:", list(minimal_batch.keys()))
print("scene_id:", minimal_batch["scene_id"], "segment_id:", minimal_batch["segment_id"])

pc_min = minimal_batch["pointcloud"]
if isinstance(pc_min, dict) and "background" in pc_min:
    bg_np = pc_min["background"]
    print("minimal pointcloud['background'] shape:", bg_np.shape)
else:
    print("minimal pointcloud type:", type(pc_min))

targets = minimal_batch["targets"]
print("num minimal targets:", len(targets))
first_target = targets[0]
print("first target frame_idx:", first_target.get("frame_idx"))
gt_image = first_target["gt_image"]
print("gt_image shape:", tuple(gt_image.shape))


In [ ]:
# Cell X: visualize point cloud with Open3D

import open3d as o3d

# Use minimal_batch from previous cells
bg_np = minimal_batch["pointcloud"]["background"]
coords_np = bg_np[:, :3].astype(np.float32)
if bg_np.shape[1] >= 6:
    colors_np = bg_np[:, 3:6].astype(np.float32)
    if colors_np.max() > 1.0 + 1e-3:
        colors_np = colors_np / 255.0
else:
    colors_np = np.ones_like(coords_np, dtype=np.float32) * 0.5

pcd = o3d.geometry.PointCloud()
pcd.points = o3d.utility.Vector3dVector(coords_np)
pcd.colors = o3d.utility.Vector3dVector(colors_np)

# Optional: downsample for speed
if len(pcd.points) > 200000:
    pcd = pcd.random_down_sample(sampling_ratio=200000 / float(len(coords_np)))

print(pcd)
o3d.visualization.draw_geometries([pcd], window_name="MinimalStreetForward PointCloud")


In [ ]:
# Cell 5: visualize point cloud XYZ extents and GT image

import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

bg = minimal_batch["pointcloud"]["background"]
coords = bg[:, :3]
colors = bg[:, 3:] if bg.shape[1] >= 6 else None

print("coords min:", coords.min(axis=0))
print("coords max:", coords.max(axis=0))
print("cfg.model.bbx_min:", cfg.model.get("bbx_min"))
print("cfg.model.bbx_max:", cfg.model.get("bbx_max"))

fig = plt.figure(figsize=(6, 5))
ax = fig.add_subplot(111, projection="3d")
sample_idx = np.random.choice(coords.shape[0], size=min(5000, coords.shape[0]), replace=False)
xs, ys, zs = coords[sample_idx].T
if colors is not None and colors.shape[1] >= 3:
    cs = colors[sample_idx]
    if cs.max() > 1.0 + 1e-3:
        cs = cs / 255.0
else:
    cs = "blue"
ax.scatter(xs, ys, zs, c=cs, s=1)
ax.set_xlabel("X")
ax.set_ylabel("Y")
ax.set_zlabel("Z")
ax.set_title("Background point cloud (sampled)")
plt.show()

gt_image = minimal_batch["targets"][0]["gt_image"].detach().cpu().numpy()
plt.figure(figsize=(5, 5))
plt.imshow(np.clip(gt_image, 0.0, 1.0))
plt.axis("off")
plt.title("GT image (target 0)")
plt.show()


In [ ]:
# Cell 6: build MinimalStreetForward model

from models.streetforward.minimal_trainer import MinimalStreetForward

model = MinimalStreetForward(config=cfg, device=device).to(device)
model.eval()

# move tensors in minimal_batch to device where needed
minimal_batch_device = dict(minimal_batch)
minimal_batch_device["targets"] = [dict(minimal_batch["targets"][0])]
if isinstance(minimal_batch_device["targets"][0]["gt_image"], torch.Tensor):
    minimal_batch_device["targets"][0]["gt_image"] = minimal_batch_device["targets"][0]["gt_image"].to(device)

print("Model built on device:", device)


In [ ]:
# Cell 7: point cloud -> 3D feature volume -> per-point features

with torch.no_grad():
    means, anchor_rgb = model._pointcloud_to_means_rgb(minimal_batch["pointcloud"])
    means = means.to(device)
    anchor_rgb = anchor_rgb.to(device)

print("means shape:", means.shape)
print("means min:", means.min(dim=0).values)
print("means max:", means.max(dim=0).values)

# inspect sparse tensor and volume
sparse_feat, vol_dim, valid_coords = model.construct_sparse_tensor(
    raw_coords=means.clone(),
    feats=anchor_rgb,
    Bbx_min=model.bbx_min,
    Bbx_max=model.bbx_max,
    voxel_size=model.voxel_size,
    device=device,
)
print("vol_dim:", vol_dim)
print("valid_coords shape:", valid_coords.shape)
if valid_coords.shape[0] > 0:
    print("valid_coords min:", valid_coords.min(dim=0).values)
    print("valid_coords max:", valid_coords.max(dim=0).values)

feat_3d = model.sparse_conv(sparse_feat)
dense_volume = model.sparse_to_dense_volume(
    sparse_tensor=feat_3d,
    coords=valid_coords,
    vol_dim=vol_dim,
).unsqueeze(0)
dense_volume = dense_volume.permute(0, 4, 3, 2, 1)  # [1, C, D, H, W]
print("dense_volume shape:", dense_volume.shape)

from models.streetforward.minimal_trainer import _get_grid_coords, _interpolate_features

grid_coords = _get_grid_coords(
    position_w=means,
    bbx_min=model.bbx_min,
    bbx_max=model.bbx_max,
    vol_dim=vol_dim,
    voxel_size=model.voxel_size,
)
print("grid_coords shape:", grid_coords.shape)
print("grid_coords min:", grid_coords.min(dim=0).values)
print("grid_coords max:", grid_coords.max(dim=0).values)

feat_3d_crop = _interpolate_features(grid_coords, dense_volume)
print("feat_3d_crop shape:", feat_3d_crop.shape)


In [ ]:
# Cell 8: 3DGS parameter prediction

with torch.no_grad():
    render_params = model._predict_render_params(means, anchor_rgb, feat_3d_crop)

for k, v in render_params.items():
    if isinstance(v, torch.Tensor) and v.numel() > 0:
        print(k, v.shape, "min", float(v.min().item()), "max", float(v.max().item()))
    else:
        print(k, v.shape)

opacities = render_params["opacities_r"]
print("opacities stats: min", float(opacities.min().item()),
      "max", float(opacities.max().item()),
      "mean", float(opacities.mean().item()))

from models.streetforward.math_utils import _sh_to_rgb

sh_dc = render_params["colors_r"][:, 0, :]
rgb_from_sh = _sh_to_rgb(sh_dc)
print("rgb_from_sh stats: min", rgb_from_sh.min(dim=0).values,
      "max", rgb_from_sh.max(dim=0).values)


In [ ]:
# Cell 9: single-view render vs GT comparison

import torch.nn.functional as F

target0 = minimal_batch_device["targets"][0]
view = target0["view"]
gt_image_dev = target0["gt_image"].to(device)
H, W = gt_image_dev.shape[0], gt_image_dev.shape[1]

with torch.no_grad():
    pred_rgb, acc = model._render_single_view(render_params, view, height=H, width=W)

pred_np = torch.clamp(pred_rgb.detach().cpu(), 0.0, 1.0).numpy()
gt_np = torch.clamp(gt_image_dev.detach().cpu(), 0.0, 1.0).numpy()
l1 = float(F.l1_loss(torch.from_numpy(pred_np), torch.from_numpy(gt_np)).item())
mse = float(((pred_np - gt_np) ** 2).mean())

print(f"L1 loss (manual): {l1:.6f}")
print(f"MSE (manual): {mse:.6e}")

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(pred_np)
axes[0].set_title("Pred RGB (manual pipeline)")
axes[0].axis("off")
axes[1].imshow(gt_np)
axes[1].set_title("GT image")
axes[1].axis("off")
plt.show()


In [ ]:
# Cell 10: compare manual pipeline with model.forward

batch_for_forward = {
    **minimal_batch,
    "targets": [dict(minimal_batch["targets"][0])],
}
batch_for_forward["targets"][0]["gt_image"] = batch_for_forward["targets"][0]["gt_image"].to(device)

with torch.no_grad():
    out = model.forward(batch_for_forward)

pred_fwd = out["pred_rgb"].detach().cpu()
gt_fwd = out["gt_image"].detach().cpu()

print("pred_fwd shape:", tuple(pred_fwd.shape))
print("gt_fwd shape:", tuple(gt_fwd.shape))

max_diff = float((pred_fwd - pred_rgb.detach().cpu()).abs().max().item())
print(f"max |pred_fwd - pred_manual|: {max_diff:.6e}")

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(torch.clamp(pred_fwd, 0.0, 1.0).numpy())
axes[0].set_title("Pred RGB (model.forward)")
axes[0].axis("off")
axes[1].imshow(torch.clamp(pred_rgb.detach().cpu(), 0.0, 1.0).numpy())
axes[1].set_title("Pred RGB (manual)")
axes[1].axis("off")
plt.show()


In [ ]:
# Cell 11: camera pose and simple projection sanity checks

from models.streetforward.math_utils import get_viewmat

target0 = minimal_batch["targets"][0]
view = target0["view"]
c2w = view.camtoworlds if hasattr(view, "camtoworlds") else view["camtoworlds"]
print("Camera-to-world (c2w):")
print(c2w)

viewmat = get_viewmat(c2w)
print("World-to-camera (viewmat):")
print(viewmat)

Ks = None
if hasattr(view, "Ks"):
    Ks = view.Ks[0]
elif hasattr(view, "K"):
    Ks = view.K[0] if view.K.dim() == 3 else view.K
else:
    Ks = torch.eye(3, device=device)
print("Intrinsics K:")
print(Ks)

c2w_cpu = c2w.detach().cpu().numpy()
cam_pos = c2w_cpu[:3, 3]

fig = plt.figure(figsize=(6, 5))
ax = fig.add_subplot(111, projection="3d")
sample_idx = np.random.choice(coords.shape[0], size=min(5000, coords.shape[0]), replace=False)
xs, ys, zs = coords[sample_idx].T
ax.scatter(xs, ys, zs, c="lightgray", s=1, alpha=0.5)
ax.scatter([cam_pos[0]], [cam_pos[1]], [cam_pos[2]], c="red", s=50, label="camera")

axes_dirs = c2w_cpu[:3, :3]
scale = 1.0
origin = cam_pos
for color, direction, name in [("r", axes_dirs[:, 0], "x"),
                               ("g", axes_dirs[:, 1], "y"),
                               ("b", axes_dirs[:, 2], "z")]:
    ax.quiver(
        origin[0], origin[1], origin[2],
        direction[0], direction[1], direction[2],
        length=scale, color=color, label=f"cam_{name}"
    )

ax.set_title("Camera pose vs point cloud")
ax.legend()
plt.show()

def project_points_world_to_image(points_world: np.ndarray,
                                  c2w_mat: torch.Tensor,
                                  K_mat: torch.Tensor) -> np.ndarray:
    """Project world points to pixel coordinates using c2w and K.

    points_world: [N, 3] numpy, world coords
    returns: [N, 2] numpy pixel coords (may go out of bounds)
    """
    N = points_world.shape[0]
    pts = torch.from_numpy(points_world).float().to(device)
    ones = torch.ones((N, 1), device=device)
    pts_h = torch.cat([pts, ones], dim=-1).T  # [4, N]

    w2c = torch.inverse(c2w_mat)
    pts_cam = (w2c @ pts_h)  # [4, N]
    pts_cam = pts_cam[:3, :]

    x = pts_cam[0, :] / (pts_cam[2, :] + 1e-8)
    y = pts_cam[1, :] / (pts_cam[2, :] + 1e-8)
    xy1 = torch.stack([x, y, torch.ones_like(x)], dim=0)  # [3, N]

    pix = (K_mat @ xy1).T  # [N, 3]
    return pix[:, :2].detach().cpu().numpy()

H, W = gt_np.shape[0], gt_np.shape[1]
sample_idx = np.random.choice(coords.shape[0], size=min(3000, coords.shape[0]), replace=False)
pts_world = coords[sample_idx]

pix = project_points_world_to_image(pts_world, c2w.to(device), Ks.to(device))
u = pix[:, 0]
v = pix[:, 1]

plt.figure(figsize=(5, 5))
plt.imshow(np.clip(gt_np, 0.0, 1.0))
plt.scatter(u, v, s=2, c="yellow", alpha=0.3)
plt.gca().invert_yaxis()
plt.title("Projected 3D points on GT image")
plt.axis("off")
plt.show()


In [ ]:
# Cell: Web point cloud visualization with PLYViewer\n\nimport numpy as np\n\ntry:\n    from tools.plyviewer import PLYViewer\nexcept ImportError as e:\n    print("PLYViewer not available:", e)\nelse:\n    # Use minimal_batch pointcloud as demo_pointcloud\n    demo_pointcloud = minimal_batch.get("pointcloud", {})\n\n    if demo_pointcloud is not None and len(demo_pointcloud) > 0:\n        # Create viewer (similar to StreetForward_Demo)\n        viewer = PLYViewer(\n            host="0.0.0.0",\n            port=7007,\n            point_size=0.01,\n            point_shape="circle",\n            auto_fallback=True,\n        )\n\n        viewer.start_viewer()\n\n        # Background point cloud\n        background = demo_pointcloud.get("background", np.zeros((0, 6), dtype=np.float32))\n        if len(background) > 0:\n            background_points = background[:, :3]\n            background_colors = background[:, 3:]\n\n            max_points_for_viz = 100000\n            if len(background_points) > max_points_for_viz:\n                indices = np.random.choice(len(background_points), max_points_for_viz, replace=False)\n                background_points = background_points[indices]\n                background_colors = background_colors[indices]\n\n            viewer.add_point_cloud(\n                points=background_points,\n                colors=background_colors,\n                name="/background",\n                visible=True,\n            )\n            print(f"  ✓ 已添加背景点云: {len(background_points):,} 点")\n\n        # Dynamic objects point clouds (if present, though minimal_batch normally has only background)\n        dynamic = demo_pointcloud.get("dynamic", {}) if isinstance(demo_pointcloud, dict) else {}\n        if len(dynamic) > 0:\n            for intid, pts in dynamic.items():\n                if len(pts) > 0:\n                    dynamic_points = pts[:, :3]\n                    dynamic_colors = pts[:, 3:]\n\n                    max_dynamic_points = 10000\n                    if len(dynamic_points) > max_dynamic_points:\n                        indices = np.random.choice(len(dynamic_points), max_dynamic_points, replace=False)\n                        dynamic_points = dynamic_points[indices]\n                        dynamic_colors = dynamic_colors[indices]\n\n                    viewer.add_point_cloud(\n                        points=dynamic_points,\n                        colors=dynamic_colors,\n                        name=f"/dynamic/object_{intid}",\n                        visible=True,\n                    )\n\n        print("\n", viewer.viewer_info)\n        demo_viewer = viewer\n    else:\n        print("minimal_batch 中没有可用的点云数据，跳过 PLYViewer 可视化")\n